In [8]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langgraph.prebuilt import tools_condition,ToolNode
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages


load_dotenv()


True

In [9]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [10]:
class researchState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]


In [11]:
# tools  
search_tool = DuckDuckGoSearchRun(region = 'us-en')

tools = [search_tool]

llm_with_tools = llm.bind_tools(tools=tools)

 

In [12]:
# nodes

def chat_node(state: researchState) -> researchState:
     
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': response}

tool_node = ToolNode(tools)

In [13]:
graph = StateGraph(researchState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)


graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()


In [14]:

output = chatbot.invoke({'messages': [HumanMessage(content= "what is stock price of apple ?")]})
print(output['messages'][-1].content)

The current stock price of Apple is $248.09 as of March 20, 2026.
